### CPU Power Regression (robust OLS)

This cell loads CPU power experiment CSVs, cleans required metrics, and fits robust OLS models to estimate how latency affects CPU power across experiments, reporting coefficients, confidence intervals, and significance decisions.


In [18]:
# Run robust OLS on CPU power to gauge latency effects across experiments.
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path

label_map = {
    'unbalanced_processing': 'Unbalanced Processing',
    'unnecessary_processing': 'Unnecessary Processing',
    'the_ramp': 'The Ramp',
    'sisyphus_db_retrieval': 'Sisyphus DB Retrieval',
    'more_is_less': 'More Is Less',
    'godclass': 'God Class',
    'excessive_dynamic_allocation': 'Excessive Dynamic Allocation',
    'circuitous_treasure_hunt': 'Circuitous Treasure Hunt',
    'one_lane_bridge': 'One Lane Bridge',
    'traffic_jam': 'Traffic Jam',
}

root = Path('./executed/2026-ICSA-PE')
power_word = 'CPU power'
rows = []
for name, label in label_map.items():
    csv_path = root / name / f"{name}.csv"
    if not csv_path.exists():
        print(f"Skip {name}: missing {csv_path}")
        continue
    df = pd.read_csv(csv_path)
    needed = {'cpu_power', 'latency', 'requests_per_s', 'cpu_utilization', 'seconds_from_start'}
    if not needed.issubset(df.columns):
        print(f"Skip {name}: missing columns {needed - set(df.columns)}")
        continue
    for col in needed:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=needed)
    df = df[(df['cpu_power'] > 0) & (df['requests_per_s'] > 0) & (df['seconds_from_start'] >= 120)]
    if df.empty:
        print(f"Skip {name}: no usable rows")
        continue
    X = sm.add_constant(df[['latency', 'requests_per_s', 'cpu_utilization']])
    y = df['cpu_power']
    ols = sm.OLS(y, X).fit()
    ols_robust = ols.get_robustcov_results(cov_type='HC3')
    names = ols_robust.model.exog_names
    if 'latency' not in names:
        print(f"Skip {name}: latency term missing")
        continue
    idx = names.index('latency')
    beta = float(ols_robust.params[idx])
    p = float(ols_robust.pvalues[idx])
    ci_low, ci_high = map(float, ols_robust.conf_int()[idx])
    r2 = float(getattr(ols_robust, 'rsquared', np.nan))
    decision = 'Reject' if p < 0.05 else 'Keep'
    arrow = '↑' if beta > 0 else '↓' if beta < 0 else '→'
    decision_display = f"{decision} {arrow}" if decision == 'Reject' else decision
    rows.append({
        'Experiment': label,
        'beta_response_time': beta,
        'ci_low': ci_low,
        'ci_high': ci_high,
        'p_response_time': p,
        'Decision': decision_display,
    })

if rows:
    table = pd.DataFrame(rows)
    table = table.rename(columns={'beta_response_time': 'beta (Response time)', 'p_response_time': 'p (Response time)'})
    order = [label_map[name] for name in label_map if any(r['Experiment'] == label_map[name] for r in rows)]
    table['Experiment'] = pd.Categorical(table['Experiment'], categories=order, ordered=True)
    table = table.sort_values('Experiment').reset_index(drop=True)
    display(table)
else:
    print('No regression results computed.')


,Experiment,beta (Response time),ci_low,ci_high,p (Response time),Decision
0,Unbalanced Processing,-0.000083,-0.000147,-0.000018,1.186003e-02,Reject ↓
1,Unnecessary Processing,-0.000002,-0.000008,0.000004,4.226767e-01,Keep
2,The Ramp,0.018475,0.017443,0.019508,1.423159e-264,Reject ↑
3,Sisyphus DB Retrieval,-0.000115,-0.000197,-0.000032,6.347959e-03,Reject ↓
4,More Is Less,-0.001140,-0.001290,-0.000991,2.625321e-50,Reject ↓
5,God Class,0.000190,0.000153,0.000227,4.692501e-24,Reject ↑
6,Excessive Dynamic Allocation,0.000017,-0.000026,0.000059,4.450934e-01,Keep
7,Circuitous Treasure Hunt,-0.000566,-0.000771,-0.000360,7.044490e-08,Reject ↓
8,One Lane Bridge,-0.000047,-0.000119,0.000024,1.944395e-01,Keep
9,Traffic Jam,0.000029,0.000023,0.000035,1.605869e-20,Reject ↑


### DRAM Power Regression (robust OLS)

This cell repeats the regression pipeline for DRAM power, adding memory usage as a predictor to measure the relationship between latency and DRAM power with robust coefficients, intervals, and decisions.


In [19]:
# Run robust OLS on DRAM power to gauge latency effects across experiments.
import pandas as pd
import numpy as np
import statsmodels.api as sm
from pathlib import Path

label_map = {
    'unbalanced_processing': 'Unbalanced Processing',
    'unnecessary_processing': 'Unnecessary Processing',
    'the_ramp': 'The Ramp',
    'sisyphus_db_retrieval': 'Sisyphus DB Retrieval',
    'more_is_less': 'More Is Less',
    'godclass': 'God Class',
    'excessive_dynamic_allocation': 'Excessive Dynamic Allocation',
    'circuitous_treasure_hunt': 'Circuitous Treasure Hunt',
    'one_lane_bridge': 'One Lane Bridge',
    'traffic_jam': 'Traffic Jam',
}

root = Path('./executed/2026-ICSA-PE')
power_word = 'DRAM power'
rows = []
for name, label in label_map.items():
    csv_path = root / name / f"{name}.csv"
    if not csv_path.exists():
        print(f"Skip {name}: missing {csv_path}")
        continue
    df = pd.read_csv(csv_path)
    needed = {'dram_w', 'latency', 'requests_per_s', 'cpu_utilization', 'seconds_from_start', 'memory_usage'}
    if not needed.issubset(df.columns):
        print(f"Skip {name}: missing columns {needed - set(df.columns)}")
        continue
    for col in needed:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=needed)
    df = df[(df['dram_w'] >= 0) & (df['requests_per_s'] > 0) & (df['seconds_from_start'] >= 120)]
    if df.empty:
        print(f"Skip {name}: no usable rows")
        continue
    X = sm.add_constant(df[['latency', 'requests_per_s', 'cpu_utilization', 'memory_usage']])
    y = df['dram_w']
    ols = sm.OLS(y, X).fit()
    ols_robust = ols.get_robustcov_results(cov_type='HC3')
    names = ols_robust.model.exog_names
    if 'latency' not in names:
        print(f"Skip {name}: latency term missing")
        continue
    idx = names.index('latency')
    beta = float(ols_robust.params[idx])
    p = float(ols_robust.pvalues[idx])
    ci_low, ci_high = map(float, ols_robust.conf_int()[idx])
    r2 = float(getattr(ols_robust, 'rsquared', np.nan))
    decision = 'Reject' if p < 0.05 else 'Keep'
    arrow = '↑' if beta > 0 else '↓' if beta < 0 else '→'
    decision_display = f"{decision} {arrow}" if decision == 'Reject' else decision
    rows.append({
        'Experiment': label,
        'beta_response_time': beta,
        'ci_low': ci_low,
        'ci_high': ci_high,
        'p_response_time': p,
        'Decision': decision_display,
    })

if rows:
    table = pd.DataFrame(rows)
    table = table.rename(columns={'beta_response_time': 'beta (Response time)', 'p_response_time': 'p (Response time)'})
    order = [label_map[name] for name in label_map if any(r['Experiment'] == label_map[name] for r in rows)]
    table['Experiment'] = pd.Categorical(table['Experiment'], categories=order, ordered=True)
    table = table.sort_values('Experiment').reset_index(drop=True)
    display(table)
else:
    print('No regression results computed for DRAM power.')


,Experiment,beta (Response time),ci_low,ci_high,p (Response time),Decision
0,Unbalanced Processing,6.324357e-08,-4.248273e-07,5.513144e-07,7.995135e-01,Keep
1,Unnecessary Processing,-3.921092e-06,-5.926091e-06,-1.916093e-06,1.267427e-04,Reject ↓
2,The Ramp,-6.765384e-04,-7.426984e-04,-6.103784e-04,7.923593e-89,Reject ↓
3,Sisyphus DB Retrieval,-4.783103e-06,-7.328479e-06,-2.237727e-06,2.307194e-04,Reject ↓
4,More Is Less,-1.416141e-06,-2.034086e-06,-7.981951e-07,7.086432e-06,Reject ↓
5,God Class,-1.574928e-07,-3.676994e-07,5.271374e-08,1.419738e-01,Keep
6,Excessive Dynamic Allocation,-1.824574e-08,-8.201522e-07,7.836607e-07,9.644291e-01,Keep
7,Circuitous Treasure Hunt,5.579173e-06,-1.243429e-06,1.240178e-05,1.089847e-01,Keep
8,One Lane Bridge,4.682310e-07,-7.899425e-07,1.726404e-06,4.657438e-01,Keep
9,Traffic Jam,-2.428787e-08,-2.266472e-07,1.780714e-07,8.140158e-01,Keep


### CPU Power Residual Diagnostics

This cell evaluates CPU power regression residuals with Anderson–Darling normality and Breusch–Pagan heteroscedasticity tests across experiments, collecting p-values and keep/reject diagnostics.


In [20]:
# Check CPU power model residuals with Anderson-Darling and Breusch-Pagan tests.
import pandas as pd
import statsmodels.api as sm
from pathlib import Path
import warnings
from statsmodels.stats.diagnostic import het_breuschpagan, normal_ad

label_map = {
    'unbalanced_processing': 'Unbalanced Processing',
    'unnecessary_processing': 'Unnecessary Processing',
    'the_ramp': 'The Ramp',
    'sisyphus_db_retrieval': 'Sisyphus DB Retrieval',
    'more_is_less': 'More Is Less',
    'godclass': 'God Class',
    'excessive_dynamic_allocation': 'Excessive Dynamic Allocation',
    'circuitous_treasure_hunt': 'Circuitous Treasure Hunt',
    'one_lane_bridge': 'One Lane Bridge',
    'traffic_jam': 'Traffic Jam',
}

root = Path('./executed/2026-ICSA-PE')
rows = []
for name, label in label_map.items():
    csv_path = root / name / f"{name}.csv"
    if not csv_path.exists():
        print(f"Skip {name}: missing {csv_path}")
        continue
    df = pd.read_csv(csv_path)
    needed = {'cpu_power', 'latency', 'requests_per_s', 'cpu_utilization', 'seconds_from_start'}
    if not needed.issubset(df.columns):
        print(f"Skip {name}: missing columns {needed - set(df.columns)}")
        continue
    for col in needed:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=needed)
    df = df[(df['cpu_power'] > 0) & (df['requests_per_s'] > 0) & (df['seconds_from_start'] >= 120)]
    if df.empty:
        print(f"Skip {name}: no usable rows")
        continue
    X = sm.add_constant(df[['latency', 'requests_per_s', 'cpu_utilization']])
    y = df['cpu_power']
    ols = sm.OLS(y, X).fit()
    resid = ols.resid
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='divide by zero encountered in log', category=RuntimeWarning)
        ad_stat, ad_p = normal_ad(resid)
    bp_stat, bp_p, _, _ = het_breuschpagan(resid, ols.model.exog)
    rows.append({
        'Experiment': label,
        'AD p': ad_p,
        'AD decision': 'Reject' if ad_p < 0.05 else 'Keep',
        'BP p': bp_p,
        'BP decision': 'Reject' if bp_p < 0.05 else 'Keep',
    })

if rows:
    table = pd.DataFrame(rows)
    order = [label_map[name] for name in label_map if any(r['Experiment'] == label_map[name] for r in rows)]
    table['Experiment'] = pd.Categorical(table['Experiment'], categories=order, ordered=True)
    table = table.sort_values('Experiment').reset_index(drop=True)
    display(table)
else:
    print('No AD/BP results computed.')


,Experiment,AD p,AD decision,BP p,BP decision
0,Unbalanced Processing,0.0,Reject,4.443269e-02,Reject
1,Unnecessary Processing,0.0,Reject,2.422436e-01,Keep
2,The Ramp,0.0,Reject,5.958458e-44,Reject
3,Sisyphus DB Retrieval,0.0,Reject,0.000000e+00,Reject
4,More Is Less,0.0,Reject,0.000000e+00,Reject
5,God Class,0.0,Reject,0.000000e+00,Reject
6,Excessive Dynamic Allocation,0.0,Reject,1.989134e-102,Reject
7,Circuitous Treasure Hunt,0.0,Reject,6.734909e-10,Reject
8,One Lane Bridge,0.0,Reject,0.000000e+00,Reject
9,Traffic Jam,0.0,Reject,8.121154e-59,Reject


### DRAM Power Residual Diagnostics

This cell runs the same Anderson–Darling and Breusch–Pagan residual diagnostics for the DRAM power regression to flag deviations from normality or homoscedasticity per experiment.


In [21]:
# Check DRAM power model residuals with Anderson-Darling and Breusch-Pagan tests.
import pandas as pd
import statsmodels.api as sm
from pathlib import Path
import warnings
from statsmodels.stats.diagnostic import het_breuschpagan, normal_ad

label_map = {
    'unbalanced_processing': 'Unbalanced Processing',
    'unnecessary_processing': 'Unnecessary Processing',
    'the_ramp': 'The Ramp',
    'sisyphus_db_retrieval': 'Sisyphus DB Retrieval',
    'more_is_less': 'More Is Less',
    'godclass': 'God Class',
    'excessive_dynamic_allocation': 'Excessive Dynamic Allocation',
    'circuitous_treasure_hunt': 'Circuitous Treasure Hunt',
    'one_lane_bridge': 'One Lane Bridge',
    'traffic_jam': 'Traffic Jam',
}

root = Path('./executed/2026-ICSA-PE')
rows = []
for name, label in label_map.items():
    csv_path = root / name / f"{name}.csv"
    if not csv_path.exists():
        print(f"Skip {name}: missing {csv_path}")
        continue
    df = pd.read_csv(csv_path)
    needed = {'cpu_power', 'latency', 'requests_per_s', 'cpu_utilization', 'seconds_from_start'}
    if not needed.issubset(df.columns):
        print(f"Skip {name}: missing columns {needed - set(df.columns)}")
        continue
    for col in needed:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=needed)
    df = df[(df['cpu_power'] > 0) & (df['requests_per_s'] > 0) & (df['seconds_from_start'] >= 120)]
    if df.empty:
        print(f"Skip {name}: no usable rows")
        continue
    X = sm.add_constant(df[['latency', 'requests_per_s', 'cpu_utilization', 'memory_usage']])
    y = df['dram_w']
    ols = sm.OLS(y, X).fit()
    resid = ols.resid
    with warnings.catch_warnings():
        warnings.filterwarnings('ignore', message='divide by zero encountered in log', category=RuntimeWarning)
        ad_stat, ad_p = normal_ad(resid)
    bp_stat, bp_p, _, _ = het_breuschpagan(resid, ols.model.exog)
    rows.append({
        'Experiment': label,
        'AD p': ad_p,
        'AD decision': 'Reject' if ad_p < 0.05 else 'Keep',
        'BP p': bp_p,
        'BP decision': 'Reject' if bp_p < 0.05 else 'Keep',
    })

if rows:
    table = pd.DataFrame(rows)
    order = [label_map[name] for name in label_map if any(r['Experiment'] == label_map[name] for r in rows)]
    table['Experiment'] = pd.Categorical(table['Experiment'], categories=order, ordered=True)
    table = table.sort_values('Experiment').reset_index(drop=True)
    display(table)
else:
    print('No AD/BP results computed.')


,Experiment,AD p,AD decision,BP p,BP decision
0,Unbalanced Processing,0.0,Reject,4.359403e-03,Reject
1,Unnecessary Processing,0.0,Reject,2.776654e-121,Reject
2,The Ramp,0.0,Reject,0.000000e+00,Reject
3,Sisyphus DB Retrieval,0.0,Reject,0.000000e+00,Reject
4,More Is Less,0.0,Reject,3.273515e-01,Keep
5,God Class,0.0,Reject,6.938953e-01,Keep
6,Excessive Dynamic Allocation,0.0,Reject,4.250406e-01,Keep
7,Circuitous Treasure Hunt,0.0,Reject,2.187757e-22,Reject
8,One Lane Bridge,0.0,Reject,5.371874e-02,Keep
9,Traffic Jam,0.0,Reject,1.811793e-06,Reject


### Latency–Power Correlations

This cell computes Pearson and Spearman correlations between latency and CPU/DRAM power for each experiment, comparing sign agreement to highlight consistent or divergent monotonic versus linear trends.


In [22]:
# Compute correlation between latency and CPU/DRAM power for each experiment.
import pandas as pd
from pathlib import Path

label_map = {
    'unbalanced_processing': 'Unbalanced Processing',
    'unnecessary_processing': 'Unnecessary Processing',
    'the_ramp': 'The Ramp',
    'sisyphus_db_retrieval': 'Sisyphus DB Retrieval',
    'more_is_less': 'More Is Less',
    'godclass': 'God Class',
    'excessive_dynamic_allocation': 'Excessive Dynamic Allocation',
    'circuitous_treasure_hunt': 'Circuitous Treasure Hunt',
    'one_lane_bridge': 'One Lane Bridge',
    'traffic_jam': 'Traffic Jam',
}
order = list(label_map.keys())
root = Path('./executed/2026-ICSA-PE')
rows = []
for name in order:
    label = label_map[name]
    csv_path = root / name / f"{name}.csv"
    if not csv_path.exists():
        print(f"Skip {name}: missing {csv_path}")
        continue
    df = pd.read_csv(csv_path, usecols=lambda c: c in {'cpu_power', 'latency', 'dram_w'})
    for col in ['cpu_power', 'latency', 'dram_w']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['cpu_power', 'latency'])
    df = df[df['cpu_power'] >= 0]
    if df.empty:
        print(f"Skip {name}: no usable rows")
        continue
    cpu_pearson = df[['cpu_power', 'latency']].corr(method='pearson').loc['cpu_power', 'latency']
    cpu_spearman = df[['cpu_power', 'latency']].corr(method='spearman').loc['cpu_power', 'latency']
    cpu_agree = (cpu_pearson >= 0) == (cpu_spearman >= 0)
    dram_pearson = dram_spearman = float('nan'); dram_agree = False
    if 'dram_w' in df.columns:
        dram_df = df.dropna(subset=['dram_w'])
        dram_df = dram_df[dram_df['dram_w'] >= 0]
        if not dram_df.empty:
            dram_pearson = dram_df[['dram_w', 'latency']].corr(method='pearson').loc['dram_w', 'latency']
            dram_spearman = dram_df[['dram_w', 'latency']].corr(method='spearman').loc['dram_w', 'latency']
            dram_agree = (dram_pearson >= 0) == (dram_spearman >= 0)
    rows.append({
        'Experiment': label,
        'CPU Pearson r': cpu_pearson,
        'CPU Spearman rho': cpu_spearman,
        'cpu_agree': cpu_agree,
        'DRAM Pearson r': dram_pearson,
        'DRAM Spearman rho': dram_spearman,
        'dram_agree': dram_agree,
    })

if rows:
    table = pd.DataFrame(rows)
    order_labels = [label_map[n] for n in order if any(r['Experiment'] == label_map[n] for r in rows)]
    table['Experiment'] = pd.Categorical(table['Experiment'], categories=order_labels, ordered=True)
    table = table.sort_values('Experiment').reset_index(drop=True)
    display(table.drop(columns=['cpu_agree','dram_agree']))
else:
    print('No correlation results computed.')


,Experiment,CPU Pearson r,CPU Spearman rho,DRAM Pearson r,DRAM Spearman rho
0,Unbalanced Processing,0.797347,0.093249,-0.113008,-0.071553
1,Unnecessary Processing,0.203709,-0.100752,0.351697,0.786025
2,The Ramp,0.663668,0.691333,0.720682,0.747726
3,Sisyphus DB Retrieval,0.613190,-0.264050,0.641344,-0.263818
4,More Is Less,0.525505,-0.094354,0.111713,-0.017692
5,God Class,0.586526,0.045034,0.067718,0.043098
6,Excessive Dynamic Allocation,0.743343,0.061766,-0.108469,-0.028193
7,Circuitous Treasure Hunt,0.347271,0.012787,0.260962,0.017768
8,One Lane Bridge,0.648286,0.029217,-0.048552,-0.040087
9,Traffic Jam,0.296187,0.225914,-0.095529,-0.157697


### Latency and Power Summary Stats

This cell summarizes descriptive statistics (mean/max) for latency, CPU power, and DRAM power per experiment to provide context on the observed value ranges.


In [23]:
# Summarize latency and power descriptive stats for each experiment.
import pandas as pd
from pathlib import Path

label_map = {
    'unbalanced_processing': 'Unbalanced Processing',
    'unnecessary_processing': 'Unnecessary Processing',
    'the_ramp': 'The Ramp',
    'sisyphus_db_retrieval': 'Sisyphus DB Retrieval',
    'more_is_less': 'More Is Less',
    'godclass': 'God Class',
    'excessive_dynamic_allocation': 'Excessive Dynamic Allocation',
    'circuitous_treasure_hunt': 'Circuitous Treasure Hunt',
    'one_lane_bridge': 'One Lane Bridge',
    'traffic_jam': 'Traffic Jam',
}

root = Path('./executed/2026-ICSA-PE')
rows = []
for name, label in label_map.items():
    csv_path = root / name / f"{name}.csv"
    if not csv_path.exists():
        print(f"Skip {name}: missing {csv_path}")
        continue
    df = pd.read_csv(csv_path)
    for col in ['cpu_power', 'latency', 'dram_w']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['cpu_power', 'latency'])
    df = df[df['cpu_power'] >= 0]
    cpu_mean, cpu_min, cpu_max = df['cpu_power'].mean(), df['cpu_power'].min(), df['cpu_power'].max()
    lat_mean, lat_min, lat_max = df['latency'].mean(), df['latency'].min(), df['latency'].max()
    dram_mean = dram_min = dram_max = float('nan')
    if 'dram_w' in df.columns:
        dram_series = pd.to_numeric(df['dram_w'], errors='coerce')
        dram_series = dram_series[dram_series >= 0].dropna()
        if not dram_series.empty:
            dram_mean, dram_min, dram_max = dram_series.mean(), dram_series.min(), dram_series.max()
    rows.append({
        'Experiment': label,
        'Response time mean (ms)': round(lat_mean), 'Response time max (ms)': round(lat_max), #'Latency min (ms)': lat_min
        'CPU mean (W)': round(cpu_mean,2), 'CPU max (W)': round(cpu_max,2), #'CPU min (W)': cpu_min        
        'DRAM mean (W)': round(dram_mean,2), 'DRAM min (W)': round(dram_min,2), 'DRAM max (W)': round(dram_max,2),        
    })

if rows:
    table = pd.DataFrame(rows)
    order = [label_map[name] for name in label_map if any(r['Experiment'] == label_map[name] for r in rows)]
    table['Experiment'] = pd.Categorical(table['Experiment'], categories=order, ordered=True)
    table = table.sort_values('Experiment').reset_index(drop=True)
    display(table)
else:
    print('No stats computed.')


,Experiment,Response time mean (ms),Response time max (ms),CPU mean (W),CPU max (W),DRAM mean (W),DRAM min (W),DRAM max (W)
0,Unbalanced Processing,12833,14000,18.74,23.98,0.29,0.28,0.91
1,Unnecessary Processing,6728,11000,17.39,22.51,0.83,0.29,1.65
2,The Ramp,49,94,6.21,12.18,0.50,0.29,1.32
3,Sisyphus DB Retrieval,2185,2500,16.81,22.40,0.76,0.29,1.07
4,More Is Less,2289,2800,18.08,23.26,0.31,0.29,0.43
5,God Class,13693,17000,13.93,22.41,0.30,0.29,0.62
6,Excessive Dynamic Allocation,7747,8200,19.36,24.74,0.29,0.28,0.69
7,Circuitous Treasure Hunt,229,390,15.92,22.10,0.40,0.29,1.34
8,One Lane Bridge,3985,4600,18.88,23.86,0.30,0.28,1.20
9,Traffic Jam,4139,10000,17.48,23.40,0.30,0.28,1.05
